# Instalar Dependencias

In [2]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 \
  --force-reinstall --no-cache-dir

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 58.4 MB/s  0:00:14
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 246.2 MB/s  0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.16.0
    Uninstalling typing_extensions-4.16.0:
      Successfully uninstalled typing_extensions-4.16.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
  Attempting uninstall: MarkupSafe
    Found existing installation: MarkupSafe 3.0.4
    Uninstalling MarkupSafe-3.0.4:
      Successfully uninstalled MarkupSafe-3.0.4
  Attempting uninstall: diskcache
    Found existing installation: diskcache 5.6.3
    Uninstalling diskcache-5.6.3:
      Successfully uninstalled diskcache-5.6.3
  Attempting uninstall: jinja2
    Found existing installation: Jinja2

In [1]:
!pip install "numpy<2.3,>=1.24"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 37.0 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.4
    Uninstalling numpy-2.5.4:
      Successfully uninstalled numpy-2.5.4


# Descarga de Modelo

In [1]:
import os
import ctypes

cuda_path = "/usr/local/lib/python3.13/dist-packages/nvidia/cuda_runtime/lib"
os.environ["LD_LIBRARY_PATH"] = f"{cuda_path}:{os.environ.get('LD_LIBRARY_PATH', '')}"

# Carga explícita para que ctypes la mantenga en memoria
ctypes.CDLL(f"{cuda_path}/libcudart.so.12")

os.makedirs("models", exist_ok=True)
os.makedirs("data", exist_ok=True)

# Descargar modelo GGUF (aprox. 2 GB, descarga en segundos en la red de Google)
!curl -L -o models/llama-3.2-3b-instruct.Q4_K_M.gguf \
    https://huggingface.co/bartowski/Llama-3.2-3B-Instruct-GGUF/resolve/main/Llama-3.2-3B-Instruct-Q4_K_M.gguf

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   956  100   956    0     0   5045      0 --:--:-- --:--:-- --:--:--  5058
100 1925M  100 1925M    0     0   203M      0  0:00:09  0:00:09 --:--:--  239M


# Creación de Base de Conocimiento Local


In [2]:
import json

knowledge_base = [
    {
        "id": "kb_01",
        "topic": "Restablecimiento de Contraseña",
        "text": "Para restablecer tu contraseña, dirígete a https://auth.portal.local/reset, ingresa tu correo corporativo y haz clic en 'Generar enlace temporal'. El enlace expira en 15 minutos."
    },
    {
        "id": "kb_02",
        "topic": "Configuración de VPN Corporativa",
        "text": "La VPN corporativa requiere el cliente WireGuard o OpenVPN. Descarga el perfil '.ovpn' desde el portal de TI, importa el archivo e inicia sesión con credenciales de Directorio Activo agregando el token MFA de 6 dígitos al final de tu contraseña."
    },
    {
        "id": "kb_03",
        "topic": "Solicitud de Licencias de Software",
        "text": "Las licencias de software (Office, JetBrains, Figma) deben solicitarse mediante aprobación del líder de área completando el formulario TI-802. El tiempo de resolución es de 24 a 48 horas hábiles."
    }
]

with open("data/knowledge_base.json", "w", encoding="utf-8") as f:
    json.dump(knowledge_base, f, ensure_ascii=False, indent=2)

print("Base de conocimiento creada localmente en /content/data/knowledge_base.json")

Base de conocimiento creada localmente en /content/data/knowledge_base.json


# Definir el Motor RAG y la API FastAPI

Este código carga el LLM directamente en la GPU (usando n_gpu_layers=-1 para volcar todas las capas en la VRAM de la T4) y calcula la similitud con NumPy en memoria

In [3]:
%%bash
# Busca todas las carpetas 'lib' dentro de los paquetes de nvidia y las registra en el sistema
find /usr/local/lib/python3.13/dist-packages/nvidia/ -type d -name "lib" > /etc/ld.so.conf.d/nvidia_all.conf
ldconfig

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/

In [4]:
!echo "/usr/local/lib/python3.13/dist-packages/nvidia/cuda_runtime/lib" > /etc/ld.so.conf.d/cuda_runtime.conf
!ldconfig

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/

In [4]:
import os

# Agrega las rutas de bibliotecas dinámicas de CUDA al entorno
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda-12/lib64:/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

# Verifica que libcudart.so esté disponible
import ctypes
try:
    ctypes.CDLL("libcudart.so.12")
    print("✓ CUDA 12 cargado correctamente.")
except Exception as e:
    print("Error al cargar CUDA:", e)

✓ CUDA 12 cargado correctamente.


In [5]:
!pip install fastembed

In [6]:
import os
import json
import ctypes
import uuid
import numpy as np
from pydantic import BaseModel, Field
from fastembed import TextEmbedding

# Cargar librerías compartidas de CUDA antes de importar llama_cpp
for lib in ["libcudart.so.12", "libcublas.so.12", "libcublasLt.so.12"]:
    try:
        ctypes.CDLL(lib)
    except Exception:
        pass

from llama_cpp import Llama
from fastapi import FastAPI, HTTPException

# 1. Cargar Embedding en memoria
print("Cargando modelo de embeddings...")
embed_model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

# 2. Cargar LLM con aceleración GPU CUDA
print("Cargando LLM en VRAM...")
llm = Llama(
    model_path="models/llama-3.2-3b-instruct.Q4_K_M.gguf",
    n_ctx=2048,
    n_gpu_layers=-1,
    verbose=False
)

# 3. Vectorización en memoria RAM (Zero Database)
with open("data/knowledge_base.json", "r", encoding="utf-8") as f:
    docs = json.load(f)

memory_kb_texts = [f"{d['topic']}: {d['text']}" for d in docs]
memory_kb_vectors = np.array(list(embed_model.embed(memory_kb_texts)))
print(f"Base de conocimiento cargada: {len(memory_kb_texts)} artículos indexados en memoria.")

def search_context(query: str, top_k: int = 2) -> list[str]:
    query_vector = np.array(list(embed_model.embed([query]))[0])
    norms_kb = np.linalg.norm(memory_kb_vectors, axis=1)
    norm_query = np.linalg.norm(query_vector)
    similarities = np.dot(memory_kb_vectors, query_vector) / (norms_kb * norm_query)
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [memory_kb_texts[i] for i in top_indices if similarities[i] > 0.35]

# 4. Esquemas FastAPI
class TicketRequest(BaseModel):
    user_id: str = Field(..., example="usr_102")
    subject: str = Field(..., example="Problema con acceso VPN")
    description: str = Field(..., example="No puedo conectarme con mis credenciales normales.")

class TicketResponse(BaseModel):
    ticket_id: str
    category: str
    suggested_resolution: str
    context_used: list[str]

# 5. Instancia FastAPI
app = FastAPI(title="Google Colab AI Helpdesk")

@app.post("/api/v1/tickets/process", response_model=TicketResponse)
def process_ticket(ticket: TicketRequest):
    full_query = f"{ticket.subject} {ticket.description}"
    contexts = search_context(full_query)
    context_str = "\n".join([f"- {c}" for c in contexts]) if contexts else "Sin documentación específica disponible."

    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Eres un agente técnico de Nivel 1 para Helpdesk corporativo. Responde en español de forma concisa y estructurada.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Información de soporte disponible:
{context_str}

Ticket del usuario:
Asunto: {ticket.subject}
Descripción: {ticket.description}

Proporciona la categoría del problema y los pasos de solución:<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""
    output = llm(prompt, max_tokens=350, temperature=0.1, stop=["<|eot_id|>"])
    resolution = output["choices"][0]["text"].strip()

    category = "General"
    for cat in ["VPN", "Contraseña", "Licencias", "Hardware"]:
        if cat.lower() in full_query.lower():
            category = cat
            break

    return TicketResponse(
        ticket_id=f"TICK-{uuid.uuid4().hex[:6].upper()}",
        category=category,
        suggested_resolution=resolution,
        context_used=contexts
    )

Cargando modelo de embeddings...
Cargando LLM en VRAM...
Base de conocimiento cargada: 3 artículos indexados en memoria.


/tmp/ipykernel_9356/1068827103.py:50: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  user_id: str = Field(..., example="usr_102")
/tmp/ipykernel_9356/1068827103.py:51: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  subject: str = Field(..., example="Problema con acceso VPN")
/tmp/ipykernel_9356/1068827103.py:52: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be

# Exponer la API

In [8]:
!pip install pyngrok nest-asyncio uvicorn

In [7]:
# Terminar cualquier proceso escuchando en el puerto 8000
!fuser -k 8000/tcp

# Cerrar túneles previos de ngrok para evitar duplicados
from pyngrok import ngrok
ngrok.kill()

In [9]:
import threading
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

# 1. Configurar tu token y cerrar sesiones previas
NGROK_AUTH_TOKEN = "3LUR64okJdiWgaIKdzZpM6i1vNU_7Pa1ak49CxQG5g34fwjRK"
# ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()

# 2. Iniciar el servidor Uvicorn en segundo plano
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# 3. Conectar el túnel hacia el puerto 8000
public_url = ngrok.connect(8000).public_url

print("=" * 60)
print(f"URL PÚBLICA DE LA API: {public_url}")
print(f"DOCUMENTACIÓN SWAGGER: {public_url}/docs")
print("=" * 60)

URL PÚBLICA DE LA API: https://comfy-unsteady-conceded.ngrok-free.dev
DOCUMENTACIÓN SWAGGER: https://comfy-unsteady-conceded.ngrok-free.dev/docs


# Probar la integración directamente

In [10]:
import requests

payload = {
    "user_id": "usr_dev_44",
    "subject": "Error al conectar la VPN",
    "description": "Descargué el archivo ovpn pero al poner mi clave no me deja entrar."
}

# Hacemos la petición a localhost (o a public_url)
response = requests.post("http://127.0.0.1:8000/api/v1/tickets/process", json=payload)
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

/usr/local/lib/python3.13/dist-packages/llama_cpp/llama.py:1314: RuntimeWarning: Detected duplicate leading "<|begin_of_text|>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{
  "ticket_id": "TICK-5F6934",
  "category": "VPN",
  "suggested_resolution": "**Categoría del problema:** Conexión VPN\n\n**Pasos de solución:**\n\n1. **Verifica la configuración del archivo .ovpn**: Asegúrate de que el archivo .ovpn descargado desde el portal de TI esté configurado correctamente y que no haya errores en la configuración.\n2. **Revisa la contraseña**: Asegúrate de que la contraseña ingresada sea correcta y que incluya el token MFA de 6 dígitos agregado al final.\n3. **Verifica la conexión**: Intenta iniciar sesión nuevamente con la VPN y verifica que la conexión se establezca correctamente.\n4. **Contacta con el Directorio Activo**: Si el problema persiste, contacta con el Directorio Activo para obtener asistencia adicional y verificar si hay algún problema con tu cuenta o la VPN en general.\n\n**Nota:** Si el problema persiste, es posible que debas contactar con el equipo de TI para obtener asistencia adicional.",
  "context_used": [
    "Configuración de VPN Corpor